# Cargar los datos

In [53]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd() / 'data'
df = pd.read_csv(DATA_DIR / 'movies_metadata.csv', low_memory=False)
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce').fillna(0.0)
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce').fillna(0.0)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


# calcular la media de C

In [54]:
C = df['vote_average'].mean()

# Calcular numero de votos

In [55]:
m = df['vote_count'].quantile(0.90)
print(m)

160.0


In [56]:
q_movies = df.copy().loc[df['vote_count'] >= m]

# Crear columna score para almacenar media ponderada

In [57]:
# media ponderada
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']

    # Cálculo de IMDB
    return (v/(v+m) * R) + (m/(m+v) * C)

In [58]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [59]:
q_movies = q_movies.sort_values('score', ascending=False)

#Mostrar los primeros 15 resultados
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445855
834,The Godfather,6024.0,8.5,8.425420
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421309
12481,The Dark Knight,12269.0,8.3,8.265467
2843,Fight Club,9678.0,8.3,8.256373
292,Pulp Fiction,8670.0,8.3,8.251392
522,Schindler's List,4436.0,8.3,8.206613
23673,Whiplash,4376.0,8.3,8.205378
5481,Spirited Away,3968.0,8.3,8.196026
2211,Life Is Beautiful,3643.0,8.3,8.187140


# Recomendadores basados en contenido

In [60]:
df['overview'].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: str

# Vectorización TF-IDF

In [61]:
#Importar TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Definir el objeto de la clase TF-IDF Vectorizer. Quitamos stop words de inglés
tfidf = TfidfVectorizer(stop_words='english')

#Reemplazar NaN por string vacío
df['overview'] = df['overview'].fillna('')

# Consruir la matriz TF-IDF haciendo ajustes y transformaciones
tfidf_matrix = tfidf.fit_transform(df['overview'])

#Mostrar shape
tfidf_matrix.shape

(45466, 75827)

In [62]:
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [63]:
from sklearn.metrics.pairwise import linear_kernel

# Calcular la matriz de similitud coseno
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [64]:
cosine_sim.shape

(45466, 45466)

# Mapeo inverso

In [65]:
#Construct a reverse map of indices and movie titles
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

# Función de recomendación

In [66]:
# Toma el título y devuelve las 10 películas más similares y populares.
def get_recommendations(title, cosine_sim=cosine_sim, candidate_pool=30, top_n=10):
    match = indices.get(title)
    if match is None:
        raise ValueError(f'No se encontró la película: {title}')
    idx = int(match.iloc[0] if isinstance(match, pd.Series) else match)

    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda item: item[1], reverse=True)
    movie_indices = [movie_idx for movie_idx, _ in sim_scores[1:candidate_pool + 1]]
    candidates = df.iloc[movie_indices].copy()
    candidates['score'] = candidates.apply(weighted_rating, axis=1)

    return (candidates.sort_values('score', ascending=False)
            .head(top_n)[['title', 'vote_count', 'vote_average', 'score']]
            .reset_index(drop=True))

In [67]:
get_recommendations('The Dark Knight Rises')

,title,vote_count,vote_average,score
0,The Dark Knight,12269.0,8.3,8.265467
1,Batman Begins,7511.0,7.5,7.460735
2,"Batman: The Dark Knight Returns, Part 2",426.0,7.9,7.276782
3,"Batman: The Dark Knight Returns, Part 1",410.0,7.7,7.115429
4,Batman: Under the Red Hood,459.0,7.6,7.087552
5,The Lego Batman Movie,1473.0,7.2,7.044945
6,Batman,2145.0,7.0,6.904032
7,Batman: Mask of the Phantasm,218.0,7.4,6.645488
8,Batman Beyond: Return of the Joker,152.0,7.5,6.534598
9,Batman: Year One,255.0,7.1,6.528421


In [68]:
get_recommendations('The Godfather')

,title,vote_count,vote_average,score
0,The Godfather: Part II,3418.0,8.3,8.180043
1,The Godfather: Part III,1589.0,7.1,6.964377
2,American Movie,57.0,7.7,6.164491
3,Live by Night,416.0,6.2,6.038185
4,Election,51.0,6.7,5.879121
5,The Godfather Trilogy: 1972-1990,13.0,8.8,5.856616
6,Blood Ties,175.0,6.0,5.817297
7,Eulogy,34.0,6.4,5.754611
8,Miss Bala,34.0,6.2,5.719559
9,The Most Beautiful Wife,4.0,8.5,5.687772


# Mejorando el recomendador

In [69]:
# cargar conjuntos de datos adicionales
credits = pd.read_csv(DATA_DIR / 'credits.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords.csv')

# Eliminar algunos IDs problemáticos
df = df.drop([19730, 29503, 35587, 35803], errors='ignore')

# Convetir todos los ids a números enteros
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

# Hacer merges entre data frames
df = df.merge(credits, on='id', how='inner')
df = df.merge(keywords, on='id', how='inner')

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\carlo\\Documents\\inteligencia artificial\\recomendadores_basado_en_contenido\\data\\credits.csv'

In [ ]:
# Parse the stringified features into their corresponding Python objects.
from ast import literal_eval

def parse_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == '':
        return []
    try:
        parsed = literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []
    return parsed if isinstance(parsed, list) else []

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(parse_list)

KeyError: 'cast'

# Extracción de datos

In [ ]:
import numpy as np

In [ ]:
def get_crew_members(crew, jobs, limit=3):
    return [member.get('name', '') for member in crew
            if member.get('job') in jobs and member.get('name')][:limit]

def get_director(crew):
    directors = get_crew_members(crew, {'Director'}, limit=1)
    return directors[0] if directors else ''

def get_writers(crew):
    return get_crew_members(crew, {'Writer', 'Screenplay', 'Story'})

def get_producers(crew):
    return get_crew_members(crew, {'Producer', 'Executive Producer'})

In [ ]:
def get_list(items, limit=3):
    if isinstance(items, list):
        names = [item.get('name', '') for item in items if item.get('name')]
        #Checar si existen más de 3 elementos. Si sí, regresar primeros 3, si no, todos
        return names[:limit]

    # regresar lista vacía si los datos no están bien formateados
    return []

In [ ]:
# Extraer director, guionistas y productores de la columna crew
df['director'] = df['crew'].apply(get_director)
df['writers'] = df['crew'].apply(get_writers)
df['producers'] = df['crew'].apply(get_producers)

# Extraer top 3 de elenco, palabras clave y géneros
features = ['cast', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(get_list)

In [ ]:
df[['title', 'cast', 'director', 'writers', 'producers', 'keywords', 'genres']].head(3)

In [ ]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        #Checar si existse el director. Si no, regresar ""
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [ ]:
features = ['cast', 'keywords', 'director', 'writers', 'producers', 'genres']

for feature in features:
    df[feature] = df[feature].apply(clean_data)

In [ ]:
def create_soup(x):
    # Repetir el director le da un peso mayor en la similitud.
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + ' '.join(x['director'].split() * 3) + ' ' + ' '.join(x['writers']) + ' ' + ' '.join(x['producers']) + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)

df[['soup']].head(2)

# Vectorizador CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])


count_matrix.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

# Reset index of your main DataFrame and construct reverse mapping as before
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

In [ ]:
get_recommendations('The Dark Knight Rises', cosine_sim2)

In [ ]:
get_recommendations('The Godfather', cosine_sim2)